# Measurement plots (unfolded data vs generators)

Loads:
- unfolding ingredients/results from `unfolding-gen1-final.ipynb`
- generator spectra from `generator_predictions.ipynb`

and makes the final xsec comparison plots (same figures formerly produced by
`generator_comparison.ipynb`).


In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import sys
from pathlib import Path
import pickle

import numpy as np
import matplotlib.pyplot as plt

sys.path.append("/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana")
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from analysis_village.numucc_1p0pi.utils import plot_unfolded_result

# Notebook-only: for del_alpha, put Data + Norm. Syst. Unc. upper-left;
# remaining (model) entries lower-right in a single column.
_plot_unfolded_result_orig = plot_unfolded_result


def plot_unfolded_result(unfold, measured, models, var_config, *args, **kwargs):
    _orig_legend = plt.legend
    split_legend = var_config.var_save_name == "tki-del_alpha"

    def _legend(handles, labels, **leg_kwargs):
        if not split_legend or len(handles) < 3:
            return _orig_legend(handles, labels, **leg_kwargs)
        leg_data = _orig_legend(
            handles[:2], labels[:2],
            loc="upper left", fontsize=12, frameon=False, ncol=1,
        )
        plt.gca().add_artist(leg_data)
        return _orig_legend(
            handles[2:], labels[2:],
            loc="lower right", bbox_to_anchor=(1.0, 0.04),
            fontsize=12, frameon=False, ncol=1,
        )

    plt.legend = _legend
    try:
        return _plot_unfolded_result_orig(
            unfold, measured, models, var_config, *args, **kwargs
        )
    finally:
        plt.legend = _orig_legend


In [3]:
DATA_RESULTS = Path("/exp/sbnd/data/users/munjung/xsec/RESULTS/DATA_RESULTS")
UNFOLD_PKL = DATA_RESULTS / "gen1_final_unfold" / "unfolding_ingredients_and_results.pkl"
PRED_PKL = DATA_RESULTS / "gen1_final_unfold" / "generator_predictions.pkl"

save_fig = True
save_fig_dir = Path("/exp/sbnd/data/users/munjung/plots/numucc1p0pi/generator_comparison-PRL")
save_fig_dir.mkdir(parents=True, exist_ok=True)
print("unfold :", UNFOLD_PKL)
print("preds  :", PRED_PKL)
print("plots  :", save_fig_dir)


unfold : /exp/sbnd/data/users/munjung/xsec/RESULTS/DATA_RESULTS/gen1_final_unfold/unfolding_ingredients_and_results.pkl
preds  : /exp/sbnd/data/users/munjung/xsec/RESULTS/DATA_RESULTS/gen1_final_unfold/generator_predictions.pkl
plots  : /exp/sbnd/data/users/munjung/plots/numucc1p0pi/generator_comparison-PRL


In [4]:
with open(UNFOLD_PKL, "rb") as f:
    unfold_pack = pickle.load(f)
with open(PRED_PKL, "rb") as f:
    pred_pack = pickle.load(f)

XSEC_UNIT = float(unfold_pack["meta"]["xsec_unit"])
print("variables in unfold:", sorted(unfold_pack["variables"]))
print("generators:", list(pred_pack["generators"]))
print("xsec_unit:", XSEC_UNIT)


FileNotFoundError: [Errno 2] No such file or directory: '/exp/sbnd/data/users/munjung/xsec/RESULTS/DATA_RESULTS/gen1_final_unfold/generator_predictions.pkl'

In [ ]:
VC_BY_NAME = {
    "integrated": VariableConfig.all_events(),
    "muon-p": VariableConfig.muon_momentum(),
    "muon-dir_z": VariableConfig.muon_direction(),
    "proton-p": VariableConfig.proton_momentum(),
    "proton-dir_z": VariableConfig.proton_direction(),
    "tki-del_Tp": VariableConfig.tki_del_Tp(),
    "tki-del_Tp_x": VariableConfig.tki_del_Tp_x(),
    "tki-del_Tp_y": VariableConfig.tki_del_Tp_y(),
    "tki-del_p": VariableConfig.tki_del_p(),
    "tki-del_alpha": VariableConfig.tki_del_alpha(),
    "tki-del_phi": VariableConfig.tki_del_phi(),
}


def unfold_dict_for_plot(blob):
    r = blob["results"]
    return {
        "unfold": r["unfold"],
        "AddSmear": r["AddSmear"],
        "UnfoldCov": r["UnfoldCov"],
        "StatUnfoldCov": r["StatUnfoldCov"],
        "SystUnfoldCov": r["SystUnfoldCov"],
    }


def spectrum_or_none(gen_label, vsn):
    gens = pred_pack["generators"]
    if gen_label not in gens:
        return None
    return gens[gen_label]["spectra_by_save_name"].get(vsn)


## Data vs GENIE AR23 only


In [ ]:
PLOT_VARS = [
    "tki-del_Tp",
    "tki-del_alpha",
    "tki-del_phi",
    "muon-p",
    "muon-dir_z",
    "proton-p",
    "proton-dir_z",
]

for vsn in PLOT_VARS:
    if vsn not in unfold_pack["variables"]:
        print("skip missing", vsn)
        continue
    blob = unfold_pack["variables"][vsn]
    vc = VC_BY_NAME[vsn]
    models = {"GENIE AR23_20i": [blob["results"]["model"], "C0"]}
    save_name = str(save_fig_dir / f"{vsn}-xsec")
    plot_unfolded_result(
        unfold_dict_for_plot(blob),
        blob["results"]["measured"],
        models,
        vc,
        xsec_unit=XSEC_UNIT,
        save_fig=save_fig,
        save_name=save_name,
        textloc=[0.5, 0.9],
        approval="",
        data=True,
    )
    print("wrote", save_name)


## Data vs external generators (GiBUU / NEUT / NuWro)


In [ ]:
for vsn in PLOT_VARS:
    if vsn not in unfold_pack["variables"]:
        continue
    blob = unfold_pack["variables"][vsn]
    vc = VC_BY_NAME[vsn]
    models = {"GENIE AR23_20i": [blob["results"]["model"], "C0"]}
    for label, color in [
        ("GiBUU 2025", "C1"),
        ("NEUT 6.1.4", "C2"),
        ("NuWro 25.11.1", "C3"),
    ]:
        spec = spectrum_or_none(label, vsn)
        if spec is None:
            continue
        models[label] = [np.asarray(spec, dtype=float), color]

    save_name = str(save_fig_dir / f"{vsn}-xsec_comparison-generators")
    plot_unfolded_result(
        unfold_dict_for_plot(blob),
        blob["results"]["measured"],
        models,
        vc,
        xsec_unit=XSEC_UNIT,
        save_fig=save_fig,
        save_name=save_name,
        textloc=[0.5, 0.9],
        approval="",
        data=True,
    )
    print("wrote", save_name)


## AR25 / HF variant comparisons (variables with flat predictions)


In [ ]:
for vsn in ["tki-del_Tp", "tki-del_alpha", "tki-del_phi"]:
    if vsn not in unfold_pack["variables"]:
        continue
    blob = unfold_pack["variables"][vsn]
    vc = VC_BY_NAME[vsn]

    models_ar25 = {"GENIE AR23_20i": [blob["results"]["model"], "C0"]}
    for label, color in [
        ("GENIE AR25_20i", "C8"),
        ("GENIE AR25_20i MINERvA FA", "C4"),
        ("GENIE AR25_20i LQCD FA", "C5"),
    ]:
        spec = spectrum_or_none(label, vsn)
        if spec is not None:
            models_ar25[label] = [np.asarray(spec, dtype=float), color]
    save_name = str(save_fig_dir / f"{vsn}-xsec_comparison-AR25")
    plot_unfolded_result(
        unfold_dict_for_plot(blob),
        blob["results"]["measured"],
        models_ar25,
        vc,
        xsec_unit=XSEC_UNIT,
        save_fig=save_fig,
        save_name=save_name,
        textloc=[0.5, 0.9],
        approval="",
        data=True,
    )
    print("wrote", save_name)

    models_hf = {"GENIE AR23_20i": [blob["results"]["model"], "C0"]}
    for label, color in [
        ("GENIE G21_11a HF", "firebrick"),
        ("GENIE G21_11a HF-CRPA", "olivedrab"),
    ]:
        spec = spectrum_or_none(label, vsn)
        if spec is not None:
            models_hf[label] = [np.asarray(spec, dtype=float), color]
    save_name = str(save_fig_dir / f"{vsn}-xsec_comparison-HF")
    plot_unfolded_result(
        unfold_dict_for_plot(blob),
        blob["results"]["measured"],
        models_hf,
        vc,
        xsec_unit=XSEC_UNIT,
        save_fig=save_fig,
        save_name=save_name,
        textloc=[0.5, 0.9],
        approval="",
        data=True,
    )
    print("wrote", save_name)
